In [ ]:
%pip install google-cloud-vision google-auth google-api-core

In [3]:
#import gcp credidentals+ required libraries
import os , argparse, glob
import math
from collections import Counter
from google.cloud import vision
import re
import pandas as pd

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] ='gcpkey.json'
WORD = re.compile(r"\w+")

In [ ]:
def detect_text(path):
    """Detects text in the file."""

    client = vision.ImageAnnotatorClient()

    with open(path, "rb") as image_file:
        content = image_file.read()

    image = vision.Image(content=content)

    # for non-dense text 
    # response = client.text_detection(image=image)
    # for dense text
    response = client.document_text_detection(image=image)
    texts = response.text_annotations
    ocr_text = []

    for text in texts:
        ocr_text.append(f"\r\n{text.description}")

    if response.error.message:
        raise Exception(
            "{}\nFor more info on error messages, check: "
            "https://cloud.google.com/apis/design/errors".format(response.error.message)
        )
    return ocr_text

In [5]:
# Resolve project directory and image path robustly (no GCP calls here)
import os
from pathlib import Path

# Determine a base directory that contains gcpkey.json (expected in 'ocr/' folder)
candidate_dirs = [Path.cwd(), Path.cwd() / "ocr"]
base_dir = next((d for d in candidate_dirs if (d / "gcpkey.json").exists()), None)
if base_dir is None:
    # Fallback to current working directory if key not found (still try to build image path)
    base_dir = Path.cwd()

# Set GOOGLE_APPLICATION_CREDENTIALS to an absolute path if the file exists
cred_path = base_dir / "gcpkey.json"
if cred_path.exists():
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(cred_path)

# Build the image path relative to the resolved base_dir
image_path = base_dir / "training_materials" / "photo-1.jpg"

# Quick sanity output
print("cwd:", Path.cwd())
print("Resolved base_dir:", base_dir)
print("Resolved image_path:", image_path)
print("Image exists:", image_path.exists())

cwd: /workspaces/UWE-AI-Project/ocr
Resolved base_dir: /workspaces/UWE-AI-Project/ocr
Resolved image_path: /workspaces/UWE-AI-Project/ocr/training_materials/photo-1.jpg
Image exists: True


In [6]:
from pathlib import Path

# Use the resolved image_path from the previous cell and verify it exists
p = Path(image_path) if 'image_path' in globals() else None
if not p or not p.exists():
    parent = p.parent if p else Path.cwd()
    available = "\n".join(str(x) for x in sorted(parent.glob("*")))
    raise FileNotFoundError(
        f"Image not found at: {p if p else '(image_path not defined)'}\n"
        f"Checked directory: {parent}\n"
        f"Available files:\n{available}"
    )

# Run OCR
text = detect_text(str(p))

# Preview a few lines
print(text[:5])

["\r\n湖南家常菜\nHUNAN STIR FRIED VEGETABLES\n□ 湖南小炒香干 $15.9\nStir-Fried Hunan Smoked Tofu\n□ 川香辣子鸡 $18.8\n□土匪猪肝 $18.9\nSichuan-Style Spicy Chicken\nHunan-Style Pork Liver\n□家烧糯香茄子 $13.9\nHome-Style Braised Eggplant\n□ 酸辣鸡杂$15.9\nSour and Spicy Chicken Gizzards\n□ 生态土钵鸡 $21.8\nClaypot Kampung Chicken\n□椒茄子皮蛋 $13.9\nStir-Fried Carrots and Cured Meat\n□ 手撕包菜 $12.9\nStir-Fried Hand-Torn Cabbage\n□ 菜籽油焖豆腐 $19.8\nVegetable Oil-Braised Tofu\n□ 猪耳尖炒脆藕条 $188\nStried Polars with Crispy Lotus foot Strips\n□青椒豆辣炒油渣$13.9\nStir-Fried Green Peppers and Beans with Co\n□ 梅菜扣肉 $19.8\n□ 小炒安吉脆笋 $15.9\nBraised Pork Bely with Preserved Vegetables\nStir-Fried Anji Bamboo Shoots\n□外婆菜炒豌豆米$13.9 香酥鲈鱼$19.9\n□ 麻婆豆腐$9.9\nStir-Fried Grandes Pickles and Green Peas\nCrispy Sea Bass\nMapo Tofu\n不辣也好吃\nNOT SPICY BUT ALSO DELICIOUS\nOlive Vegetable, Minced Pork and\nCommon Beans\n□ 橄榄菜肉末四季豆$12.9 番茄炒土鸡蛋$12.8\n□猪油渣机白菜$89\nStir-Fried Tomatoes and Kampung Eggs\nSt Fried Hangzhou Cobbage\nwith Crocking\n□清炒油麦菜 $1.9\nStir-Fried 

## Extract Training Data: Chinese-English Dish Pairs (in case of non-correct data)

Parse OCR output to create translation training data in format:
```
[translate:中文菜名]	English Dish Name
```

In [7]:
import re
from typing import List, Tuple

SKIP_PATTERNS = [
    r'^湖南家常菜', r'^不辣也好吃', r'^营养汤品', r'^主食\s*/\s*小吃', r'^酒水饮料',
    r'^HUNAN', r'^NOT\s+SPICY', r'^NUTRITIONAL', r'^STAPLE', r'^DRINKS', r'^\s*$'
]
PRICE_DOLLAR = re.compile(r'\$\s*\d{1,4}(?:\.\d{1,2})?')
PRICE_BARE   = re.compile(r'(?<![%\dA-Za-z年])\d{1,4}\.\d{1,2}(?!\s*%)')
HAS_ZH       = re.compile(r'[\u4e00-\u9fff]')
HAS_EN       = re.compile(r'[A-Za-z]{3,}')

def is_skip(line: str) -> bool:
    return any(re.match(p, line, re.IGNORECASE) for p in SKIP_PATTERNS)

def clean_prices(s: str) -> str:
    return PRICE_BARE.sub('', PRICE_DOLLAR.sub('', s))

def normalize_en(s: str) -> str:
    s = s.replace("St Fried", "Stir-Fried").replace("Cobbage", "Cabbage").replace("Bely", "Belly")
    s = re.sub(r'\s+', ' ', s).strip()
    return s[:1].upper() + s[1:] if s else s





In [8]:
# 2) Robust pairing from a single full OCR string
def extract_dish_pairs_from_text(full_text: str, window: int = 3) -> List[Tuple[str, str]]:
    lines = [ln.strip().replace('\r','') for ln in full_text.split('\n') if ln.strip()]
    zh_idx = [i for i,l in enumerate(lines) if (not is_skip(l)) and HAS_ZH.search(l)]
    en_idx = [i for i,l in enumerate(lines) if (not is_skip(l)) and (not HAS_ZH.search(l)) and HAS_EN.search(l)]

    used_en = set()
    pairs: List[Tuple[str, str]] = []
    for ci in zh_idx:
        zh_raw = lines[ci]
        zh = clean_prices(re.sub(r'^□\s*', '', zh_raw))
        zh = re.sub(r'\s+', '', zh).strip()
        if not zh or not HAS_ZH.search(zh):
            continue

        # nearest English within ±window
        best_e, best_d = None, 999
        for ei in en_idx:
            if ei in used_en:
                continue
            d = abs(ei - ci)
            if d <= window and d < best_d:
                best_e, best_d = ei, d

        if best_e is not None:
            en = normalize_en(clean_prices(lines[best_e]))
            if en:
                pairs.append((zh, en))
                used_en.add(best_e)

    # dedupe exact pairs
    out, seen = [], set()
    for zh, en in pairs:
        if (zh, en) not in seen:
            out.append((zh, en)); seen.add((zh, en))
    return out

In [13]:
full_text = detect_text(image_path)                   # uses your helper from above
dish_pairs = extract_dish_pairs_from_text(full_text)  # robust pairing

print(f"Extracted {len(dish_pairs)} dish pairs")

df = pd.DataFrame(dish_pairs, columns=["Chinese", "English"]).drop_duplicates().reset_index(drop=True)
display(df.head(20))  # optional in notebook

# 4) Export in AutoML-ready TSV
output_path = Path.cwd() / "menu_training_data_clean.tsv"
with open(output_path, 'w', encoding='utf-8') as f:
    for zh, en in df.itertuples(index=False):
        f.write(f"[translate:{zh}]\t{en}\n")

                
print(f"Saved {len(df)} rows → {output_path}")

Extracted 48 dish pairs


,Chinese,English
0,湖南小炒香干,Stir-Fried Hunan Smoked Tofu
1,川香辣子鸡,Sichuan-Style Spicy Chicken
2,家烧糯香茄子,Home-Style Braised Eggplant
3,酸辣鸡杂,Sour and Spicy Chicken Gizzards
4,生态土钵鸡,Claypot Kampung Chicken
5,椒茄子皮蛋,Stir-Fried Carrots and Cured Meat
6,手撕包菜,Stir-Fried Hand-Torn Cabbage
7,菜籽油焖豆腐,Vegetable Oil-Braised Tofu
8,猪耳尖炒脆藕条,Stried Polars with Crispy Lotus foot Strips
9,青椒豆辣炒油渣,Stir-Fried Green Peppers and Beans with Co


Saved 48 rows → /workspaces/UWE-AI-Project/ocr/menu_training_data_clean.tsv
